# Base Mini Profiling Plots

This notebook reads `base_mini_all_experiments.tsv` and creates the profiling plots requested for the BERT-mini base experiments.

It uses only Python standard library `csv`, `numpy`, and `matplotlib`. Generated figures are saved under `figures/` in this same folder.

Tested locally with `/home/thu/miniforge3/envs/IPLAB/bin/python`. If your notebook UI asks for a kernel, choose an environment with `numpy` and `matplotlib` installed.

In [ ]:
from pathlib import Path
import csv

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

# Make the notebook robust when launched from this directory or from another cwd.
BASE_DIR = Path.cwd()
if not (BASE_DIR / "base_mini_all_experiments.tsv").exists():
    BASE_DIR = Path("/home/thu/TiC-SAT/transformer_profiling/base_mini")

DATA_PATH = BASE_DIR / "base_mini_all_experiments.tsv"
FIG_DIR = BASE_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.grid": True,
    "grid.alpha": 0.28,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

metric_cols = [
    "sim_seconds", "instructions", "ops", "cpu_cycles", "memory_references",
    "load_instructions", "store_instructions", "dcache_demand_accesses",
    "dcache_demand_misses", "icache_demand_accesses", "icache_demand_misses",
    "l2_demand_accesses", "l2_demand_misses", "committed_branches",
    "branch_mispredictions", "ipc", "cpi", "dcache_demand_miss_rate",
    "icache_demand_miss_rate", "l2_demand_miss_rate", "branch_misprediction_rate",
    "btb_hit_ratio",
]
int_cols = {"d_q", "d_seq", "d_model", "num_head", "d_ff", "n_learners", "sve_bits", "row_index", "dump_index"}

def parse_number(value):
    if value == "" or value is None:
        return np.nan
    try:
        return float(value)
    except ValueError:
        return value

rows = []
with DATA_PATH.open(newline="") as fh:
    reader = csv.DictReader(fh, delimiter="	")
    for row in reader:
        for col in metric_cols:
            row[col] = parse_number(row.get(col, ""))
        for col in int_cols:
            if row.get(col, "") != "":
                row[col] = int(float(row[col]))
        row["exp_num"] = int(row["exp_id"][1:])
        rows.append(row)

rows.sort(key=lambda r: (r["exp_num"], r["row_index"]))
interval_rows = [r for r in rows if r["row_kind"] == "interval_delta"]
final_rows = sorted([r for r in rows if r["row_kind"] == "final_total"], key=lambda r: r["exp_num"])

def experiment_label(row):
    if str(row["implementation"]).startswith("Dense"):
        return f'{row["exp_id"]}\nDense\nSVE{row["sve_bits"]}'
    return f'{row["exp_id"]}\nN{row["n_learners"]} CB{row["codebook_size"]}\nSVE{row["sve_bits"]}'

for row in final_rows:
    row["experiment_label"] = experiment_label(row)
label_by_exp = {row["exp_id"]: row["experiment_label"] for row in final_rows}

exp_ids = [row["exp_id"] for row in final_rows]
exp_labels = [row["experiment_label"] for row in final_rows]
x = np.arange(len(final_rows))
dense = next(row for row in final_rows if row["exp_id"] == "E01")

stage_order = [
    "MHA",
    "Projection",
    "non_GEMM_after_projection",
    "FF1",
    "FF2",
    "non_GEMM_after_ff2",
]
stage_labels = {
    "MHA": "MHA",
    "Projection": "Projection",
    "non_GEMM_after_projection": "Post projection",
    "FF1": "FF1",
    "FF2": "FF2",
    "non_GEMM_after_ff2": "Post FF2",
}
stage_colors = {
    "MHA": "#4C78A8",
    "Projection": "#F58518",
    "non_GEMM_after_projection": "#54A24B",
    "FF1": "#E45756",
    "FF2": "#72B7B2",
    "non_GEMM_after_ff2": "#B279A2",
}

def values(metric):
    return np.array([float(row[metric]) for row in final_rows], dtype=float)

def human_count(value, _pos=None):
    value = float(value)
    sign = "-" if value < 0 else ""
    value = abs(value)
    for unit in ["", "K", "M", "B", "T"]:
        if value < 1000:
            return f"{sign}{value:g}{unit}"
        value /= 1000
    return f"{sign}{value:g}P"

def finish_axis(ax, title, ylabel=None, xlabel=None, rotate=True):
    ax.set_title(title, pad=10, weight="bold")
    if ylabel:
        ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.set_xticks(x)
    ax.set_xticklabels(exp_labels, rotation=45 if rotate else 0, ha="right" if rotate else "center")
    ax.margins(x=0.02)

print(f"Loaded {len(final_rows)} experiments and {len(interval_rows)} interval rows from {DATA_PATH}")


In [ ]:
# Quick data check: final totals used by the line charts.
print(f"{'Exp':<4} {'Config':<18} {'simSeconds':>11} {'CPI':>9} {'IPC':>9} {'Instructions':>15} {'Ops':>15} {'Cycles':>15}")
for row in final_rows:
    print(
        f"{row['exp_id']:<4} "
        f"{label_by_exp[row['exp_id']].replace(chr(10), ' / '):<18} "
        f"{row['sim_seconds']:>11.6f} "
        f"{row['cpi']:>9.6f} "
        f"{row['ipc']:>9.6f} "
        f"{row['instructions']:>15.0f} "
        f"{row['ops']:>15.0f} "
        f"{row['cpu_cycles']:>15.0f}"
    )


## 1. Simulated Time by Stage

Stacked bars compare all experiments. Bar colors represent the six profiled regions.

In [ ]:
stage_time = {exp_id: {stage: 0.0 for stage in stage_order} for exp_id in exp_ids}
for row in interval_rows:
    stage_time[row["exp_id"]][row["interval"]] += float(row["sim_seconds"])

fig, ax = plt.subplots(figsize=(15, 6))
bottom = np.zeros(len(exp_ids))
for stage in stage_order:
    y = np.array([stage_time[exp_id][stage] for exp_id in exp_ids], dtype=float)
    ax.bar(
        x,
        y,
        bottom=bottom,
        label=stage_labels[stage],
        color=stage_colors[stage],
        edgecolor="white",
        linewidth=0.5,
    )
    bottom += y

finish_axis(ax, "Simulated time by profiling stage", "simSeconds")
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, 1.20), frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_sim_seconds_stacked_by_stage.png", bbox_inches="tight")
plt.show()


## 2. Memory Access and Miss Comparison

This line chart uses final totals. The y-axis is logarithmic because memory references, accesses, and misses differ by orders of magnitude.

In [ ]:
memory_metrics = [
    ("memory_references", "Memory refs", "#2F4B7C"),
    ("dcache_demand_accesses", "L1D accesses", "#4C78A8"),
    ("dcache_demand_misses", "L1D misses", "#A0CBE8"),
    ("icache_demand_accesses", "L1I accesses", "#59A14F"),
    ("icache_demand_misses", "L1I misses", "#8CD17D"),
    ("l2_demand_accesses", "L2 accesses", "#E15759"),
    ("l2_demand_misses", "L2 misses", "#FF9D9A"),
]

fig, ax = plt.subplots(figsize=(15, 6))
for metric, label, color in memory_metrics:
    ax.plot(x, values(metric), marker="o", linewidth=2, markersize=4, label=label, color=color)

finish_axis(ax, "Memory references, cache accesses, and cache misses", "Count, log scale")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(FuncFormatter(human_count))
ax.legend(ncols=4, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_memory_access_and_misses.png", bbox_inches="tight")
plt.show()


## 3. Branch Prediction Comparison

The left axis shows branch counts. The right axis shows rates.

In [ ]:
fig, ax1 = plt.subplots(figsize=(15, 6))

count_lines = [
    ("committed_branches", "Committed branches", "#4C78A8"),
    ("branch_mispredictions", "Branch mispredictions", "#E15759"),
]
for metric, label, color in count_lines:
    ax1.plot(x, values(metric), marker="o", linewidth=2, markersize=4, label=label, color=color)

finish_axis(ax1, "Branch prediction behavior", "Count, log scale")
ax1.set_yscale("log")
ax1.yaxis.set_major_formatter(FuncFormatter(human_count))

ax2 = ax1.twinx()
ax2.plot(
    x,
    values("branch_misprediction_rate"),
    marker="s",
    linestyle="--",
    linewidth=2,
    markersize=4,
    label="Misprediction rate",
    color="#F28E2B",
)
ax2.plot(
    x,
    values("btb_hit_ratio"),
    marker="^",
    linestyle="--",
    linewidth=2,
    markersize=4,
    label="BTB hit ratio",
    color="#59A14F",
)
ax2.set_ylabel("Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
ax2.grid(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, ncols=4, loc="upper center", bbox_to_anchor=(0.5, 1.22), frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_branch_prediction.png", bbox_inches="tight")
plt.show()


## 4. CPI, Instructions, Ops, and CPU Cycles

Each subplot compares the raw metric on the left axis and the corresponding multiple versus the E01 dense baseline on the right axis.

In [ ]:
compute_metrics = [
    ("cpi", "CPI", "CPI"),
    ("instructions", "Instructions", "Count"),
    ("ops", "Ops", "Count"),
    ("cpu_cycles", "CPU cycles", "Cycles"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
axes = axes.ravel()

for ax, (metric, title, ylabel) in zip(axes, compute_metrics):
    y = values(metric)
    baseline = float(dense[metric])
    dense_multiple = y / baseline

    ax.plot(x, y, marker="o", linewidth=2.2, color="#4C78A8", label=title)
    finish_axis(ax, title, ylabel)
    if metric != "cpi":
        ax.yaxis.set_major_formatter(FuncFormatter(human_count))

    ax2 = ax.twinx()
    ax2.plot(x, dense_multiple, marker="s", linestyle="--", linewidth=1.9, color="#F58518", label="x dense")
    ax2.axhline(1.0, color="#666666", linewidth=1, linestyle=":", alpha=0.75)
    ax2.set_ylabel("Multiple vs E01 dense")
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="best", frameon=False)

fig.suptitle("Core metrics and dense-baseline multiples", y=1.02, weight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "04_cpi_insts_ops_cycles_vs_dense.png", bbox_inches="tight")
plt.show()


## Figure Files

Run the cells above to regenerate these files:

- `figures/01_sim_seconds_stacked_by_stage.png`
- `figures/02_memory_access_and_misses.png`
- `figures/03_branch_prediction.png`
- `figures/04_cpi_insts_ops_cycles_vs_dense.png`